In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df= pd.read_csv("D:\\Pooja_Khatri_Water_Forecasting\\brisbane water quality dataset\\brisbane_water_quality.csv\\brisbane_water_quality.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
# Convert 'Timestamp' column to datetime if necessary
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [ ]:
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df.nunique()

In [ ]:
print(df["Chlorophyll [quality]"].unique())

In [ ]:
 quality_cols = [col for col in df.columns if col.strip().endswith('[quality]')]

 for col in quality_cols:
        print(f"Unique values in '{col}':")
        print(df[col].unique())
        print('-' * 40)

In [ ]:
# Drop columns with quality information
df = df.drop(columns=['Dissolved Oxygen [quality]', 'Chlorophyll [quality]', 'Temperature [quality]',
                      'Dissolved Oxygen (%Saturation) [quality]', 'pH [quality]', 'Salinity [quality]',
                      'Specific Conductance [quality]', 'Turbidity [quality]'], errors='ignore')

In [ ]:
df.shape

In [ ]:

# Number of fully duplicated rows
num_duplicate_rows = df.duplicated().sum()
print("Number of duplicate rows:", num_duplicate_rows)

In [ ]:
print("A) Starting shape:", df.shape)

# how many timestamps are repeated?
duplicate_rows = df.duplicated(subset=["Timestamp"]).sum()
print("A) Duplicate Timestamp rows:", duplicate_rows)

# show a few duplicated timestamps
dups = df[df.duplicated(subset=["Timestamp"], keep=False)].sort_values("Timestamp")
print("A) Example duplicated timestamps (first 10 rows):")
print(dups[["Timestamp"]].head(10))

In [ ]:
#Merge duplicate timestamps (take mean)
df2 = df.copy()

# "Record number" is not a real sensor reading, so drop it
if "Record number" in df2.columns:
    df2 = df2.drop(columns=["Record number"])

print("\nB) Shape before merging duplicates:", df2.shape)

# group by Timestamp and take mean of numeric columns
df2 = df2.groupby("Timestamp", as_index=False).mean(numeric_only=True)

print("B) Shape after merging duplicates:", df2.shape)
print("B) Is Timestamp unique now?", df2["Timestamp"].is_unique)


In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_time_vs_col(df, col_name):
    # --- defensive checks ---
    if col_name == "Timestamp":
        return

    # ensure Timestamp is datetime
    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

    # sort and set index
    df = df.sort_values("Timestamp")
    df = df.set_index("Timestamp")

    # stats
    total_na = df[col_name].isna().sum()
    mean_val = df[col_name].mean()
    min_val = df[col_name].min()
    max_val = df[col_name].max()
    try:
        mode_val = df[col_name].mode().iloc[0]
    except IndexError:
        mode_val = "No Mode"

    title = (
        f"{col_name} | Mean: {mean_val:.2f} | "
        f"Max: {max_val:.2f} | Min: {min_val:.2f} | "
        f"Mode: {mode_val} | NaNs: {total_na}"
    )

    fig = go.Figure()

    # --- normal values ---
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[col_name],
        mode="lines",
        name=col_name,
        line=dict(color="blue")
    ))

    # --- NaN timestamps (red markers) ---
    nan_mask = df[col_name].isna()

    fig.add_trace(go.Scatter(
        x=df.index[nan_mask],
        y=[0] * nan_mask.sum(),  # baseline to make NaNs visible
        mode="markers",
        name="NaN",
        marker=dict(color="red", size=8),
        showlegend=True
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Timestamp",
        yaxis_title=col_name,
        template="plotly_white",
        height=500
    )

    fig.show()


for col in df.columns:
    if col != "Timestamp":
        plot_time_vs_col(df, col)


In [ ]:
# Ensure data is sorted by time (mandatory for time-series ops)
df2 = df2.sort_values("Timestamp").reset_index(drop=True)

# List of features to interpolate (excluding timestamp)
features = [col for col in df2.columns if col != "Timestamp"]

# Before interpolation: check missing counts
print("Missing values BEFORE interpolation:")
print(df2[features].isnull().sum())

# Apply LINEAR interpolation to each feature
df2[features] = df2[features].interpolate(method="linear", axis=0)

# After interpolation: check missing counts
print("\nMissing values AFTER interpolation:")
print(df2[features].isnull().sum())

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_time_vs_col(df, col_name):
    if col_name == "Timestamp":
        return

    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.sort_values("Timestamp")
    df = df.set_index("Timestamp")

    total_na = df[col_name].isna().sum()
    mean_val = df[col_name].mean()
    min_val = df[col_name].min()
    max_val = df[col_name].max()
    try:
        mode_val = df[col_name].mode().iloc[0]
    except IndexError:
        mode_val = "No Mode"

    title = (
        f"{col_name} | Mean: {mean_val:.2f} | "
        f"Max: {max_val:.2f} | Min: {min_val:.2f} | "
        f"Mode: {mode_val} | NaNs: {total_na}"
    )

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[col_name],
        mode="lines",
        name=col_name,
        line=dict(color="blue")
    ))

    nan_mask = df[col_name].isna()

    fig.add_trace(go.Scatter(
        x=df.index[nan_mask],
        y=[0] * nan_mask.sum(),
        mode="markers",
        name="NaN",
        marker=dict(color="red", size=8),
        showlegend=True
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Timestamp",
        yaxis_title=col_name,
        template="plotly_white",
        height=500
    )

    fig.show()

# Use df2 (interpolated dataframe) here
for col in df2.columns:
    if col != "Timestamp":
        plot_time_vs_col(df2, col)


In [ ]:
df2.describe(include='all')

In [ ]:
df2.isnull().sum()

In [ ]:
# Drop redundant columns
df3 = df2.drop(
    columns=[
        "Dissolved Oxygen (%Saturation)",
        "Specific Conductance"
    ],
    errors="ignore"  # avoids crash if column is missing
)

# Optional sanity check
print(df3.columns)

In [ ]:
df3.describe(include='all')

In [ ]:
df3.isnull().sum()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go

# ----------------------------
# Config
# ----------------------------
TARGET_COL = "Dissolved Oxygen"
TIME_COL   = "Timestamp"
SEQ_LEN    = 24
TEST_RATIO = 0.2
HORIZON    = 1  # ✅ predict DO at t+1

# ----------------------------
# Use df3
# ----------------------------
df = df3.copy()

# Ensure Timestamp is datetime + sorted
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)

# ----------------------------
# 1) Define X, y  (✅ Option A)
# X includes ALL sensor columns including DO (exclude Timestamp)
# y is future DO at t+HORIZON
# ----------------------------
input_cols = [c for c in df.columns if c != TIME_COL]  # includes TARGET_COL
X_raw = df[input_cols].values.astype(np.float32)

y_raw = df[TARGET_COL].shift(-HORIZON).values.astype(np.float32).reshape(-1, 1)

# Drop last HORIZON rows (target becomes NaN there)
X_raw = X_raw[:-HORIZON]
y_raw = y_raw[:-HORIZON]

print("Input columns (X):", input_cols)
print("X_raw shape:", X_raw.shape, "y_raw shape:", y_raw.shape)

# ----------------------------
# 2) Time-based split (no shuffle)
# ----------------------------
n = len(y_raw)  # ✅ important (not len(df))
split_idx = int(n * (1 - TEST_RATIO))

X_train_raw, X_test_raw = X_raw[:split_idx], X_raw[split_idx:]
y_train_raw, y_test_raw = y_raw[:split_idx], y_raw[split_idx:]

print("Train rows:", len(X_train_raw), "Test rows:", len(X_test_raw))

# ----------------------------
# Plot DO train vs test (target series) for sanity
# Note: y_raw corresponds to df[TARGET_COL] shifted by -HORIZON
# So timestamps align with df up to n rows
# ----------------------------
df_plot = df.iloc[:n].copy()
df_plot["DO_target_t_plus_h"] = y_raw.reshape(-1)

df_train = df_plot.iloc[:split_idx]
df_test  = df_plot.iloc[split_idx:]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_train[TIME_COL], y=df_train["DO_target_t_plus_h"],
    mode="lines", name="Train target DO(t+1)", line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=df_test[TIME_COL], y=df_test["DO_target_t_plus_h"],
    mode="lines", name="Test target DO(t+1)", line=dict(color="red")
))
fig.update_layout(
    title="Target Dissolved Oxygen over Time (Temporal split) — DO(t+1)",
    xaxis_title="Time", yaxis_title="Dissolved Oxygen",
    template="plotly_white"
)
fig.show()

# ----------------------------
# 3) Scale (fit ONLY on train to avoid leakage)
# ----------------------------
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train = x_scaler.fit_transform(X_train_raw)
X_test  = x_scaler.transform(X_test_raw)

y_train = y_scaler.fit_transform(y_train_raw)
y_test  = y_scaler.transform(y_test_raw)

# ----------------------------
# 4) Make sequences
# ----------------------------
def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32).reshape(-1, 1)

# Train sequences
X_train_seq, y_train_seq = make_sequences(X_train, y_train, SEQ_LEN)

# Test sequences: prepend last SEQ_LEN train rows for context (past only, no leakage)
X_test_ext = np.vstack([X_train[-SEQ_LEN:], X_test])
y_test_ext = np.vstack([y_train[-SEQ_LEN:], y_test])

X_test_seq, y_test_seq = make_sequences(X_test_ext, y_test_ext, SEQ_LEN)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("X_test_seq:", X_test_seq.shape)
print("y_test_seq:", y_test_seq.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

n_features = X_train_seq.shape[2]

model = tf.keras.Sequential([
    layers.Input(shape=(SEQ_LEN, n_features)),

    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.2),

    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.Huber(),
    metrics=["mae"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5),
]

history = model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    shuffle=False,
    callbacks=callbacks
)



In [ ]:
# Predict (scaled)
y_pred_scaled = model.predict(X_test_seq)

# Inverse scale
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_true = y_scaler.inverse_transform(y_test_seq)

# Flatten for metrics
y_pred = y_pred.reshape(-1)
y_true = y_true.reshape(-1)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

print("First 5 true vs pred:")
for i in range(5):
    print(y_true[i], " | ", y_pred[i])



In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.graph_objects as go

# ----------------------------
# 1) Metrics
# ----------------------------
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

print(f"MAE  (DO): {mae:.4f}")
print(f"RMSE (DO): {rmse:.4f}")
print(f"R²        : {r2:.4f}")

# ----------------------------
# 2) Correct timestamp alignment
# ----------------------------
# For Option A:
# Predictions correspond to:
# df index: split_idx + SEQ_LEN  →  n-1

start_idx = split_idx
end_idx   = split_idx + len(y_true)

test_timestamps = df.iloc[start_idx:end_idx][TIME_COL].values

print("Check lengths -> timestamps:", len(test_timestamps),
      "y_true:", len(y_true),
      "y_pred:", len(y_pred))

# ----------------------------
# 3) Plot
# ----------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test_timestamps,
    y=y_true,
    mode="lines",
    name="True DO(t+1)"
))

fig.add_trace(go.Scatter(
    x=test_timestamps,
    y=y_pred,
    mode="lines",
    name="Predicted DO(t+1)"
))

fig.update_layout(
    title="DO Forecast (t+1) — True vs Predicted",
    xaxis_title="Time",
    yaxis_title="Dissolved Oxygen",
    template="plotly_white"
)

fig.show()
